# Convolutional networks on MNIST

## Why this matters

CNNs matter because they use the image layout instead of treating pixels as unrelated numbers, so they learn from how a picture is actually arranged. An MLP ignores the layout: shuffle the pixels the same way for every image and it trains just as well.

Picture one small filter slid everywhere. A vertical-edge detector that works in one corner works in every corner, so one set of weights finds the same pattern anywhere in the image.

The key calculation is the parameter count of a conv layer, (k^2 * C_in + 1) * C_out, independent of image size. The from-scratch example is the vertical-edge filter [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], 9 weights that fire at a dark-to-bright edge. Sharing those weights across the image is why the CNN here uses about one eighth of the MLP's parameters. Pooling keeps the strongest response in each block, so small shifts inside a block do not change the answer. Convolutions are shift-equivariant; the dense head on top is not, which is where shift tolerance breaks.

In [ ]:
from common.dl import get_device, set_seed, setup_keras

setup_keras("torch")
import keras
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch

from common.datasets import load_mnist
from common.plotting import plot_history

SEED = 42
set_seed(SEED)
keras.utils.set_random_seed(SEED)
np.set_printoptions(precision=2, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)
print("device:", get_device())
torch.backends.cudnn.deterministic = True  # same seed, same numbers on a GPU

## 1. Intuition

A vertical edge looks the same wherever it sits. One filter that detects it, applied everywhere, needs 9 weights, not one per pixel per detector. Two ideas do the work:

- **Local connections.** Each output reads a small patch, not the whole image.
- **Shared weights.** One filter runs everywhere, so a pattern found in one corner is found in all.

Pooling keeps the strongest response in each block, halving the map and ignoring small shifts inside it.

## 2. Maths and cost function

For a 2D image $I$ and a $k\times k$ filter $K$, the output at $(i,j)$ is

$$O_{ij}=\sum_{a=0}^{k-1}\sum_{b=0}^{k-1}K_{ab}\,I_{i+a,\,j+b}+b_0$$

A layer with $C_{in}$ input and $C_{out}$ output channels has $(k^2C_{in}+1)\,C_{out}$ weights, independent of image size. The loss is unchanged.

## 3. From scratch

A dark-to-bright vertical edge filter over an 8x8 image, edge in the middle, max-pooled in 2x2 blocks.

In [ ]:
def conv2d(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Valid cross-correlation of one 2D image with one 2D kernel (what deep learning calls conv)."""
    kh, kw = kernel.shape
    h, w = image.shape
    out = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = (image[i : i + kh, j : j + kw] * kernel).sum()
    return out


def max_pool(x: np.ndarray, size: int = 2) -> np.ndarray:
    h, w = x.shape[0] // size, x.shape[1] // size
    return x[: h * size, : w * size].reshape(h, size, w, size).max(axis=(1, 3))


# A synthetic 8x8 float image: dark left, bright right, with a vertical edge at column 4.
# The vertical filter (rows of [-1, 0, 1]) is a hand-built vertical-edge detector,
# negative weight on the left, positive on the right, so its response is large exactly
# at the edge. Nothing is sampled here, so no seed is needed.
image = np.zeros((8, 8))
image[:, 4:] = 1.0  # a vertical edge: dark left, bright right
vertical = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
fmap = conv2d(image, vertical)
print("feature map (6x6), columns:\n", fmap[0])
print("pooled (3x3):\n", max_pool(fmap))
print("same kernel on the edge shifted one pixel right:")
shifted = np.roll(image, 1, axis=1)
shifted[:, 0] = 0
print(
    " feature map row:",
    conv2d(shifted, vertical)[0],
    "pooled row:",
    max_pool(conv2d(shifted, vertical))[0],
)
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, im, t in zip(ax, (image, fmap, max_pool(fmap)), ("image 8x8", "conv 6x6", "pool 3x3")):
    a.imshow(im, cmap="gray")
    a.set_title(t)
    a.axis("off")
plt.show()

The feature map is zero except two columns at the edge, where it is 3. Pooling keeps the 3 and shrinks the map from 6x6 to 3x3. Pooling absorbs shifts smaller than its block, not larger; section 6 returns to this.

## 4. Library version

Two conv-and-pool blocks (16 then 32 filters, 3x3), then a dense softmax.

In [ ]:
# Data: 28x28 grey digits, 5,000 train and 1,000 test, reshaped to (n, 28, 28, 1).
# x is float in [0, 1], y is the digit 0-9. seed fixes which subset is drawn.
x_tr, y_tr, x_te, y_te = load_mnist(5000, 1000, seed=SEED)
x_tr4, x_te4 = x_tr[..., None], x_te[..., None]  # (n, 28, 28, 1)


def make_cnn(filters: tuple[int, int] = (16, 32)) -> keras.Model:
    set_seed(SEED)
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential(
        [
            keras.Input((28, 28, 1)),
            keras.layers.Conv2D(filters[0], 3, activation="relu"),
            keras.layers.MaxPooling2D(2),
            keras.layers.Conv2D(filters[1], 3, activation="relu"),
            keras.layers.MaxPooling2D(2),
            keras.layers.Flatten(),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def make_mlp() -> keras.Model:
    set_seed(SEED)
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential(
        [
            keras.Input((28, 28, 1)),
            keras.layers.Flatten(),
            keras.layers.Dense(128, activation="relu"),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


cnn = make_cnn()
cnn.summary()
by_hand = (3 * 3 * 1 + 1) * 16 + (3 * 3 * 16 + 1) * 32 + (5 * 5 * 32 + 1) * 10
print("parameters by hand:", by_hand, "Keras:", cnn.count_params())

Hand count and Keras agree on 12,810: 160 in the first conv, 4,640 in the second, 8,010 in the dense layer. The MLP from notebook 01 has 101,770, about 8 times more. Train both for 5 epochs on the same 5,000 images.

In [ ]:
mlp = make_mlp()
h_mlp = mlp.fit(x_tr4, y_tr, epochs=5, batch_size=128, validation_split=0.2, verbose=0)
h_cnn = cnn.fit(x_tr4, y_tr, epochs=5, batch_size=128, validation_split=0.2, verbose=0)
for name, m in (("MLP", mlp), ("CNN", cnn)):
    print(
        f"{name}: {m.count_params():>7} parameters, test accuracy {m.evaluate(x_te4, y_te, verbose=0)[1]:.3f}"
    )
plot_history(h_cnn.history, ("loss", "val_loss"), title="CNN loss")
plt.show()

The first conv layer learned 16 small filters. Show them as a grid of weight images.

In [ ]:
def plot_filter_grid(
    filters: "np.ndarray",
    ncols: int = 8,
    title: str = "learned first-layer filters",
) -> "matplotlib.figure.Figure":
    """Show conv-filter weights as a grid of small greyscale images.

    `filters` is the Keras Conv2D kernel of shape (fh, fw, in_channels, n_filters),
    or (n_filters, fh, fw). For the Keras layout, the first input channel is taken.
    No randomness: this only draws the weights it is handed.
    """
    if filters.ndim == 4:  # Keras (fh, fw, in_channels, n_filters) -> (n_filters, fh, fw)
        filters = filters[:, :, 0, :].transpose(2, 0, 1)
    n = filters.shape[0]
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols, nrows))
    for i, ax in enumerate(axes.ravel()):
        if i < n:
            ax.imshow(filters[i], cmap="gray")
        ax.axis("off")
    fig.suptitle(title)
    return fig


plot_filter_grid(cnn.layers[0].get_weights()[0])  # shape (3, 3, 1, 16)
plt.show()

With one eighth of the parameters the CNN scores 91.7% against 90.0% for the MLP. The gain is small here but grows with more data. Architecture can replace parameters.

## 5. Tuning

Filter counts set the width. Four sizes, same 5 epochs, all 5,000 images.

In [ ]:
for filters in [(4, 8), (8, 16), (16, 32), (32, 64)]:
    m = make_cnn(filters)
    m.fit(x_tr4, y_tr, epochs=5, batch_size=128, verbose=0)
    print(
        f"filters {filters}: {m.count_params():>6} parameters, test accuracy {m.evaluate(x_te4, y_te, verbose=0)[1]:.3f}"
    )

Accuracy climbs from 87.9% with 2,346 parameters to 94.1% with 34,826. Even the smallest CNN is within a few points of the 101,770-parameter MLP. Single runs are noisy: the 16/32 model scores 93.0% here and 91.7% above, with only 1,000 more training images changed.

## 6. What breaks

Weight sharing should let the CNN cope with a digit placed elsewhere. Shift the test digits by 0, 2, 4 and 6 pixels.

In [ ]:
def shift(images: np.ndarray, dx: int) -> np.ndarray:
    return np.roll(images, dx, axis=2)  # digits sit in the centre, so the wrap-around is blank


for dx in (0, 2, 4, 6):
    xs = shift(x_te4, dx)
    print(
        f"shift {dx} px: MLP {mlp.evaluate(xs, y_te, verbose=0)[1]:.3f}, CNN {cnn.evaluate(xs, y_te, verbose=0)[1]:.3f}"
    )

At 2 pixels the MLP drops from 90% to 63% and the CNN from 92% to 78%. At 4 pixels the CNN is at 39%; at 6 pixels the MLP is at 9% and the CNN at 16%, both near chance (10%). The CNN does better but is not shift-proof: its convolutions are shift-equivariant, the dense classifier on top is not.

The fix: train on digits shifted by a random -4 to 4 pixels.

In [ ]:
# Augment: shift each training digit by a random -4 to 4 pixels along the width axis,
# rng seeded with SEED, to teach the model shift tolerance.
rng = np.random.default_rng(SEED)
x_aug = np.stack(
    [np.roll(im, int(d), axis=1) for im, d in zip(x_tr4, rng.integers(-4, 5, len(x_tr4)))]
)
cnn_aug = make_cnn()
cnn_aug.fit(x_aug, y_tr, epochs=5, batch_size=128, verbose=0)
for dx in (0, 4):
    print(
        f"shift {dx} px: CNN trained on shifted digits {cnn_aug.evaluate(shift(x_te4, dx), y_te, verbose=0)[1]:.3f}"
    )

A CNN uses a fraction of the weights by sharing them, yet still needs examples of every nuisance it should ignore. Next: reuse a CNN someone else trained on a far larger set.